## 1. Importação das bibliotecas


In [2]:
import os
import pandas as pd
from sqlmodel import Field, SQLModel, create_engine, select
from sqlalchemy import text
import uuid
import hashlib

## 2. Configuração inicial


In [ ]:
import os
from pathlib import Path

ROOT = Path(os.getcwd()).resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

DATA_PATH = str(ROOT / 'dados' / 'entrada') + '/'
DB_PATH = str(ROOT / 'dados' / 'banco') + '/'
os.makedirs(DB_PATH, exist_ok=True)

## 3. Funções auxiliares


In [ ]:
def _hash_id(prefix: str, value: str, length: int = 6) -> str:
    """
    Gera um ID estável e reproduzível a partir de um prefixo e um valor string.
    Usa MD5 truncado — mesmo valor sempre produz o mesmo ID, independente
    da ordem ou de reexecuções.

    Exemplos:
        _hash_id('P', 'FULANO DA SILVA')  -> 'Pa3f9c1'
        _hash_id('S', 'CTEC')             -> 'Se82b04'
    """
    digest = hashlib.md5(value.strip().upper().encode()).hexdigest()[:length]
    return prefix + digest


def insert_or_ignore(table, conn, keys, data_iter):
    """Insere linhas ignorando conflitos de PK — evita falha em reexecuções."""
    rows = [dict(zip(keys, row)) for row in data_iter]
    conn.execute(
        text(
            f"INSERT OR IGNORE INTO {table.name} "
            f"({', '.join(keys)}) "
            f"VALUES ({', '.join([':' + k for k in keys])})"
        ),
        rows,
    )

## 4. Leitura dos arquivos


In [ ]:
def filtra_arquivos(path: str) -> list:
    """
    Lê todos os arquivos presentes no diretório de entrada, filtra e retorna uma lista
    apenas com nomes de arquivo referentes a ofertas.
    """
    todos_documentos = os.listdir(DATA_PATH)
    for i in range(len(todos_documentos)):
        todos_documentos[i] = todos_documentos[i].lower()
    documentos_ofertas = list()
    for documento in todos_documentos:
        if 'oferta_20' in documento:
            documentos_ofertas.append(documento)
    documentos_ofertas = sorted(documentos_ofertas)
    return documentos_ofertas


def le_ofertas(documentos_ofertas: list, path: str) -> list:
    """
    Recebe uma lista com nomes de arquivos .xlsx (ofertas a cada período) e uma string de caminho (DATA_PATH),
    retornando uma lista de DataFrames com colunas:
    "Código", "Disciplina", "Turma", "Docente 1", "CH Docente 1", "Docente 2", "CH Docente 2",
    "Horário ", "Local ", "Matriculados", "Capacidade", "Oferta".
    """
    lista_df_ofertas = list()
    for documento in documentos_ofertas:
        df_oferta = pd.read_excel(DATA_PATH + documento)
        df_oferta['Oferta'] = documento[7:13]
        lista_df_ofertas.append(df_oferta)
    return lista_df_ofertas


def le_setores(path_to_file: str) -> pd.DataFrame:
    """
    Lê o arquivo setores_ctec.xlsm onde há a definição de setores para as disciplinas,
    retornando um DataFrame com colunas "Codigo", "Disciplina" e "Setores".
    """
    df_setores = pd.read_excel(path_to_file)
    return df_setores


for tbl in ['setores', 'disciplinas', 'professor', 'oferta']:
    SQLModel.metadata.remove(SQLModel.metadata.tables[tbl]) if tbl in SQLModel.metadata.tables else None

## 5. Modelos do banco de dados


In [ ]:
class Setores(SQLModel, table=True):
    __tablename__ = "setores"
    id_setor: str | None = Field(default=None, primary_key=True)
    nome_setor: str


class Disciplinas(SQLModel, table=True):
    codigo: str | None = Field(default=None, primary_key=True)
    nome_disciplina: str
    carga_horaria: int
    id_setor: str | None = Field(default=None, foreign_key="setores.id_setor")


class Professor(SQLModel, table=True):
    id_professor: str | None = Field(default=None, primary_key=True)
    nome_professor: str


class Oferta(SQLModel, table=True):
    id_oferta: str | None = Field(default=None, primary_key=True)
    codigo: str | None = Field(default=None, foreign_key="disciplinas.codigo")
    turma: int
    id_professor: str | None = Field(default=None, foreign_key="professor.id_professor")
    ch_professor: int
    id_professor2: str | None = Field(default=None, foreign_key="professor.id_professor")
    ch_professor2: int | None = None
    horario: str
    local: str
    matriculados: int
    capacidade: int
    oferta: str

## 6. Persistência no SQLite


In [ ]:
def grava_dados(lista_df_ofertas: list, df_setores: pd.DataFrame, DB_PATH: str) -> None:
    """
    Recebe a lista dos dfs gerados, convertendo-os para SQL e salvando-os em um banco de dados (ofertas.db)
    com a seguinte estrutura:

    Setores
    - id_setor: varchar / PK
    - nome_setor: varchar

    Disciplinas
    - codigo: varchar / PK
    - nome_disciplina: varchar
    - carga_horaria: int (derivado da soma de CH Docente 1 + CH Docente 2)
    - id_setor: FK

    Professor
    - id_professor: varchar / PK
    - nome_professor: char

    Oferta
    - id_oferta: varchar / PK
    - codigo: FK
    - turma: int
    - id_professor: FK
    - ch_professor: int
    - id_professor2: FK (null)
    - ch_professor2: int (null)
    - horario: varchar
    - local: char
    - matriculados: int
    - capacidade: int
    - oferta: str
    """

    engine = create_engine(f"sqlite:///{DB_PATH}ofertas.db")
    SQLModel.metadata.create_all(engine)
    df_ofertas = pd.concat(lista_df_ofertas, ignore_index=True)
    df_ofertas["Turma"] = df_ofertas["Turma"].str.extract(r"(\d+)").astype(int)

    # Setores
    # CORREÇÃO: IDs gerados via hash do nome — estáveis entre execuções e
    # independentes da ordem dos registros, eliminando colisão em reexecuções.
    df_setor_insert = (
        df_setores[['Setores']]
        .drop_duplicates()
        .dropna()
        .reset_index(drop=True)
        .rename(columns={'Setores': 'nome_setor'})
    )
    df_setor_insert['id_setor'] = df_setor_insert['nome_setor'].apply(
        lambda nome: _hash_id('S', nome)
    )

    df_setores = df_setores.merge(df_setor_insert, left_on='Setores', right_on='nome_setor', how='left')

    # Professores
    # CORREÇÃO: IDs gerados via hash do nome — mesmo professor sempre recebe
    # o mesmo ID, independente de quantos períodos são carregados ou da ordem
    # dos arquivos, eliminando colisão de FK em reexecuções.
    profs1 = df_ofertas[['Docente 1']].rename(columns={'Docente 1': 'nome_professor'})
    profs2 = df_ofertas[['Docente 2']].rename(columns={'Docente 2': 'nome_professor'})
    df_prof_insert = (
        pd.concat([profs1, profs2])
        .drop_duplicates()
        .dropna()
        .reset_index(drop=True)
    )
    df_prof_insert['id_professor'] = df_prof_insert['nome_professor'].apply(
        lambda nome: _hash_id('P', nome)
    )

    # Disciplinas
    # CORREÇÃO: uso de groupby + agg com critério explícito (first para nome,
    # max para carga horária) no lugar de drop_duplicates, que pegava a
    # primeira ocorrência por ordem de arquivo — comportamento indefinido
    # quando a mesma disciplina aparece em múltiplos períodos.
    df_disc_raw = df_ofertas[['Código', 'Disciplina', 'CH Docente 1', 'CH Docente 2']].copy()
    df_disc_raw['carga_horaria'] = (
        df_disc_raw['CH Docente 1'].fillna(0) + df_disc_raw['CH Docente 2'].fillna(0)
    ).astype(int)

    df_disc_insert = (
        df_disc_raw.groupby('Código', as_index=False)
        .agg(
            nome_disciplina=('Disciplina', 'first'),
            carga_horaria=('carga_horaria', 'max'),
        )
        .rename(columns={'Código': 'codigo'})
    )

    df_disc_insert = df_disc_insert.merge(
        df_setores[['Codigo', 'id_setor']], left_on='codigo', right_on='Codigo', how='left'
    )
    df_disc_insert.drop(columns=['Codigo'], inplace=True)

    # Ofertas
    df_oferta_insert = df_ofertas.copy()

    df_oferta_insert['CH Docente 1'] = df_oferta_insert['CH Docente 1'].astype(int)
    df_oferta_insert['CH Docente 2'] = pd.to_numeric(
        df_oferta_insert['CH Docente 2'], errors='coerce'
    ).astype('Int64')

    df_oferta_insert = df_oferta_insert.merge(
        df_prof_insert, left_on='Docente 1', right_on='nome_professor', how='left'
    )
    df_oferta_insert.rename(columns={'id_professor': 'id_prof1'}, inplace=True)
    df_oferta_insert.drop(columns=['nome_professor'], inplace=True)

    df_oferta_insert = df_oferta_insert.merge(
        df_prof_insert, left_on='Docente 2', right_on='nome_professor', how='left'
    )
    df_oferta_insert.rename(columns={'id_professor': 'id_prof2'}, inplace=True)
    df_oferta_insert.drop(columns=['nome_professor'], inplace=True)

    df_oferta_insert['id_oferta'] = [str(uuid.uuid4()) for _ in range(len(df_oferta_insert))]

    df_oferta_insert = df_oferta_insert[[
        'id_oferta', 'Código', 'Turma', 'id_prof1', 'CH Docente 1',
        'id_prof2', 'CH Docente 2', 'Horário ', 'Local ', 'Matriculados', 'Capacidade', 'Oferta'
    ]].rename(columns={
        'Código': 'codigo',
        'Turma': 'turma',
        'id_prof1': 'id_professor',
        'CH Docente 1': 'ch_professor',
        'id_prof2': 'id_professor2',
        'CH Docente 2': 'ch_professor2',
        'Horário ': 'horario',
        'Local ': 'local',
        'Matriculados': 'matriculados',
        'Capacidade': 'capacidade',
        'Oferta': 'oferta',
    })

    with engine.begin() as conn:
        df_setor_insert.to_sql('setores', conn, if_exists='append', index=False, method=insert_or_ignore)
        df_disc_insert.to_sql('disciplinas', conn, if_exists='append', index=False, method=insert_or_ignore)
        df_prof_insert.to_sql('professor', conn, if_exists='append', index=False, method=insert_or_ignore)
        df_oferta_insert.to_sql('oferta', conn, if_exists='append', index=False)

## 7. Execução do ETL


In [5]:
documentos_ofertas = filtra_arquivos(DATA_PATH)
lista_df_ofertas = le_ofertas(documentos_ofertas, DATA_PATH)
df_setores = le_setores(DATA_PATH + 'setores_ctec.xlsm')

if os.path.exists(DB_PATH + 'ofertas.db'):
    os.remove(DB_PATH + 'ofertas.db')

grava_dados(lista_df_ofertas, df_setores, DB_PATH)